# Tuần 5 — Train detector 4 class (player / goalkeeper / referee / ball)
Notebook chỉ gọi script; logic nằm trong `src/` và `scripts/`.

In [ ]:
# Bật GPU (T4 x2 hoặc P100) + Internet trong Settings của notebook
!git clone https://github.com/thaisanh1602/football_cv_tracking.git
%cd football_cv_tracking
!pip install -q -r requirements.txt

## Tải dataset từ Roboflow
Thêm secret `ROBOFLOW_API_KEY` (Add-ons → Secrets). Kiểm tra workspace / project / version trên trang dataset.

In [ ]:
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

rf = Roboflow(api_key=UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))
project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
project.version(1).download("yolov8", location="/kaggle/working/roboflow")  # v1: không resize (v2, v17+ bị stretch)

In [ ]:
# Kiểm tra version: ảnh phải giữ tỷ lệ 16:9 (vd 1920x1080), không bị stretch thành hình vuông
from collections import Counter
from pathlib import Path
from PIL import Image

for split in ["train", "valid", "test"]:
    imgs = list(Path(f"/kaggle/working/roboflow/{split}/images").glob("*"))
    print(split, len(imgs), Counter(Image.open(i).size for i in imgs).most_common(3))

## Đổi thứ tự class về chuẩn của repo
Roboflow xếp class theo alphabet (ball=0...), repo dùng player=0 ... ball=3.

In [ ]:
!python scripts/prepare_dataset.py --src /kaggle/working/roboflow --dst /kaggle/working/dataset

In [ ]:
!python scripts/train.py --model yolo11m.pt --imgsz 1280 --epochs 100 --batch 8 --project /kaggle/working/runs

## Lưu weights
Sau khi chạy xong: Save Version → tạo Kaggle Dataset từ output, dùng lại ở notebook eval và tải `best.pt` về máy.

In [ ]:
!cp /kaggle/working/runs/det4/weights/best.pt /kaggle/working/best.pt